In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from copy import deepcopy
from typing import Dict, List, Optional, Tuple
import sys
sys.path.append("/path/to/Vega/data")

import torch
from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch, init_empty_weights

from data.transforms import ImageTransform
from data.data_utils import pil_img2rgb, add_special_tokens
from data.drive_dataset_eval import DriveDatasetForEval
from modeling.qwen2 import Qwen2Tokenizer
from modeling import load_models

import json
import pickle
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import math

from inferencer import InterleaveInferencerAction

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
ckpt_path = "/path/to/checkpoints/#######"

model_path = "/path/to/Bagel-7B-MoT"
model, vae_model = load_models(model_path, meta=True)
model = load_checkpoint_and_dispatch(
    model,
    checkpoint=os.path.join(ckpt_path, "ema.safetensors"),
    device_map="auto",
    dtype=torch.bfloat16,
    force_hooks=True,
)
model = model.eval()
vae_model = vae_model.to("cuda")

tokenizer = Qwen2Tokenizer.from_pretrained(model_path)
tokenizer, new_token_ids, _ = add_special_tokens(tokenizer)

vae_transform = ImageTransform(256, 128, 16)
vit_transform = ImageTransform(224, 112, 14)

inferencer = InterleaveInferencerAction(
    model=model, 
    vae_model=vae_model, 
    tokenizer=tokenizer, 
    vae_transform=vae_transform, 
    vit_transform=vit_transform, 
    new_token_ids=new_token_ids
)

In [4]:
dataset_path = "/path/to/navtest_instruct.pkl"
sensor_blobs_path = "/path/to/nuplan-v1.1/sensor_blobs"

dataset = DriveDatasetForEval(
    dataset_path=dataset_path, 
    sensor_blobs_path=sensor_blobs_path, 
    history_actions="none", 
    instruction_type="rule_based_instruction",
)

In [5]:
def plot_trajectories(trajectories, xlim=5, ylim=3):
    fig, ax = plt.subplots()
    colors = ["red", "blue", "green", "yellow"]
    for i, (name, traj) in enumerate(trajectories.items()):
        ax.plot(traj[:, 0], traj[:, 1], color=colors[i], linewidth=2)
        ax.scatter(traj[:, 0], traj[:, 1], color=colors[i], label=name, s=5)

    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    ax.set_xlim(min(0, x_min), max(xlim, x_max))
    ax.set_ylim(min(-ylim, y_min), max(ylim, y_max))
    ax.set_aspect("equal")
    ax.legend(loc='upper right')

    return fig, ax


In [6]:
action_kwargs=dict(
    cfg_text_scale=4.0,
    cfg_img_scale=2.0,
    cfg_act_scale=2.0,
    cfg_interval=[0.0, 1.0],
    timestep_shift=3.0,
    num_timesteps=20,
    cfg_renorm_min=0.0,
    cfg_renorm_type="text_channel",
    enable_taylorseer=False,
)

image_kwargs = deepcopy(action_kwargs)
image_kwargs["num_timesteps"] = 50

visualize_folder="visualize"
os.makedirs(visualize_folder, exist_ok=True)

In [ ]:
# Load a sample
sample = dataset[0]
token = sample["token"]
images = sample["images"]
text = sample["text"]
action = sample["action"]


In [ ]:
# Predict actions for the next 8 steps
action_pred = inferencer.inference_action(images=images[:4], instruction=text, **action_kwargs)

traj_pred = dataset.normalizer.denormalize(action_pred.cpu()).numpy()
traj_gt = dataset.normalizer.denormalize(action.cpu()).numpy()
fig, ax = plot_trajectories({"gt": traj_gt, "pred": traj_pred})
fig.show()

In [ ]:
# Predict the future image at the end of the 8 actions
image_pred = inferencer.inference_image(images=images[:4], instruction=text, action=action_pred, **image_kwargs)
plt.imshow(image_pred)